# CRA-GQA — Kaggle Training Smoke Test

Quick end-to-end training check for the **CRA** pipeline on a tiny synthetic NextGQA subset.

**Kaggle settings**
- Accelerator: **GPU** (T4 is enough)
- Internet: **On** (downloads `roberta-base` from Hugging Face)
- Add this repo as a Kaggle dataset *or* let the notebook clone it from GitHub

Expected runtime: a few minutes.

In [ ]:
import os
import sys
from pathlib import Path

import torch

os.environ.setdefault("HF_HOME", "/kaggle/working/hf_cache")
os.environ.setdefault("TRANSFORMERS_CACHE", "/kaggle/working/hf_cache")

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Enable the GPU accelerator in Kaggle notebook settings.")

In [ ]:
from kaggle_helpers.kaggle_utils import ensure_repo, install_smoke_deps, patch_roberta_paths

REPO_ROOT = ensure_repo()
print("Repo root:", REPO_ROOT)

install_smoke_deps()
patch_roberta_paths()

In [ ]:
from kaggle_helpers.smoke_data import setup_smoke_dataset
from kaggle_helpers.kaggle_utils import load_smoke_config
from pipeline import pipeline_fns
from utils.misc import setup_seed

paths = setup_smoke_dataset(root=REPO_ROOT / "data", max_feats=8)
print("Synthetic data written:")
for name, path in paths.items():
    print(f"  {name}: {path}")

cfgs = load_smoke_config(
    cfg_path=str(REPO_ROOT / "config/CRA/CRA_NextGQA.yml"),
    running_name="kaggle/train_smoke",
    max_feats=8,
    batch_size=2,
    epochs=1,
    record_dir="/kaggle/working/output",
)

setup_seed(cfgs["misc"]["seed"])
pipeline = pipeline_fns[cfgs["optim"]["pipeline"]](cfgs)
print("Pipeline ready:", cfgs["optim"]["pipeline"])
print("Train batches:", len(pipeline.train_dataloader))
print("Val batches:", len(pipeline.val_dataloader))

In [ ]:
# Optional: sanity-check one forward/backward step before full epoch
pipeline.model.train()
batch = next(iter(pipeline.train_dataloader))
loss, acc = pipeline.get_loss(batch)
loss_value = float(loss["total_loss"].detach().cpu())
print(f"Single-step loss={loss_value:.4f}, acc={acc:.2%}")
assert loss_value == loss_value, "Loss is NaN"

pipeline.optimizer.zero_grad()
loss["total_loss"].backward()
pipeline.optimizer.step()
print("Single training step OK")

In [ ]:
pipeline.train()

ckpt_dir = Path(cfgs["stat"]["record_dir"]) / cfgs["misc"]["running_name"] / "checkpoint"
current_ckpt = ckpt_dir / "current_checkpoint.pth"
best_ckpt = ckpt_dir / "model_best.pth"

print("Checkpoint dir:", ckpt_dir)
print("Files:", sorted(p.name for p in ckpt_dir.glob("*.pth")))
assert current_ckpt.exists(), "Training did not write current_checkpoint.pth"
print("Training smoke test passed.")

## Next step

Use `model_best.pth` (or `current_checkpoint.pth`) from `/kaggle/working/output/kaggle/train_smoke/checkpoint/` in the inference smoke-test notebook.